In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm
from pathlib import Path

import torch
import torchvision
from torchvision import transforms
import torchmetrics
import pytorch_lightning as pl

from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)

os.makedirs("results", exist_ok=True)
print("Imports OK")

In [ ]:
CHECKPOINT_PATH = "/kaggle/input/datasets/YOUR_USERNAME/cardiomegaly-cnn-models/densenet121.ckpt"  # ← update
VAL_META_CSV    = "/kaggle/input/notebooks/YOUR_USERNAME/01-data-acquisition/test.csv"             # ← update
NPY_ROOT        = "/kaggle/input/notebooks/YOUR_USERNAME/01-data-acquisition/test"

BATCH_SIZE   = 64
NUM_WORKERS  = 4
THRESHOLD    = 0.57   # Default decision threshold — override with best_threshold from notebook 03

COL_GENDER   = "gender"   # e.g. 'M' / 'F'
COL_AGE      = "anchor_age"      # numeric age in years
COL_RACE     = "race"     # e.g. 'White', 'Black', 'Asian', ...
COL_LABEL    = "Cardiomegaly"    # ground-truth binary label (0 / 1)
COL_IMG_PATH = "dicom_id" # path to the .npy image file

AGE_BINS   = [0, 40, 60, 80, 120]
AGE_LABELS = ["<40", "40-60", "60-80", "80+"]

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
class CardiomegalyModel(pl.LightningModule):
    def __init__(self, weight=4.6):
        super().__init__()
        self.model = torchvision.models.densenet121(weights='DEFAULT')
        self.model.features.conv0 = torch.nn.Conv2d(
            1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
        )
        self.model.classifier = torch.nn.Linear(in_features=1024, out_features=1)
        self.register_buffer("pos_weight", torch.tensor([weight]))
        self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        self.train_acc   = torchmetrics.Accuracy(task="binary", threshold=0.5)
        self.val_acc     = torchmetrics.Accuracy(task="binary", threshold=0.5)
        self.train_auroc = torchmetrics.AUROC(task="binary")
        self.val_auroc   = torchmetrics.AUROC(task="binary")

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)


model = CardiomegalyModel.load_from_checkpoint(CHECKPOINT_PATH)
model.to(DEVICE)
model.eval()
model.freeze()
print("Model loaded from:", CHECKPOINT_PATH)

In [ ]:
meta = pd.read_csv(VAL_META_CSV)
print(f"Total validation samples: {len(meta)}")
print(meta[[COL_GENDER, COL_AGE, COL_RACE, COL_LABEL]].head())

In [ ]:
# Bin continuous age into groups
meta["age_group"] = pd.cut(
    meta[COL_AGE], bins=AGE_BINS, labels=AGE_LABELS, right=False
).astype(str)

print("\n── Gender distribution ──")
print(meta[COL_GENDER].value_counts())
print("\n── Age group distribution ──")
print(meta["age_group"].value_counts().sort_index())
print("\n── Race distribution ──")
print(meta[COL_RACE].value_counts())
print("\n── Label distribution ──")
print(meta[COL_LABEL].value_counts())

In [ ]:
val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.472], [0.301]),
])


def load_npy(path: str) -> np.ndarray:
    """Load a .npy chest X-ray and return as float32."""
    return np.load(path).astype(np.float32)


all_probs  = []
all_labels = []

with torch.no_grad():
    for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Inference"):
        label = int(row[COL_LABEL])
        img_path = f"{NPY_ROOT}/{label}/{row[COL_IMG_PATH]}.npy"
        img  = load_npy(img_path)
        img  = val_transforms(img).unsqueeze(0).to(DEVICE)
        prob = torch.sigmoid(model(img)).item()
        all_probs.append(prob)
        all_labels.append(label)

meta["prob"]  = all_probs
meta["pred"]  = (meta["prob"] >= THRESHOLD).astype(int)
print(f"Inference complete. Overall AUROC: {roc_auc_score(meta[COL_LABEL], meta['prob']):.4f}")

In [ ]:
def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray,
                    threshold: float = THRESHOLD) -> dict:
    """Return a dict of classification metrics for one group."""
    y_pred = (y_prob >= threshold).astype(int)
    n      = len(y_true)
    n_pos  = y_true.sum()

    if n == 0:
        return {}

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel() if cm.shape == (2, 2) else (cm[0, 0], 0, 0, 0)

    precision = TP / (TP + FP + 1e-9)
    recall    = TP / (TP + FN + 1e-9)   # Sensitivity / TPR
    specificity = TN / (TN + FP + 1e-9)
    fpr       = FP / (FP + TN + 1e-9)
    fnr       = FN / (FN + TP + 1e-9)
    f1        = 2 * (precision * recall) / (precision + recall + 1e-9)
    accuracy  = (TP + TN) / n

    # AUROC / AUPRC require both classes to be present
    if n_pos > 0 and n_pos < n:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
    else:
        auroc = auprc = float("nan")

    return dict(
        n=n, n_pos=int(n_pos), prevalence=n_pos / n,
        auroc=auroc, auprc=auprc,
        accuracy=accuracy,
        precision=precision, recall=recall,
        specificity=specificity,
        f1=f1, fpr=fpr, fnr=fnr,
        tp=int(TP), fp=int(FP), tn=int(TN), fn=int(FN)
    )


def subgroup_metrics(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Compute metrics for every value of group_col and return a DataFrame."""
    rows = []
    for grp, subset in df.groupby(group_col, observed=True):
        m = compute_metrics(
            subset[COL_LABEL].values,
            subset["prob"].values
        )
        m["group_col"] = group_col
        m["group"]     = str(grp)
        rows.append(m)
    return pd.DataFrame(rows).set_index("group")


print("Metric function defined.")

In [ ]:
metrics_gender    = subgroup_metrics(meta, COL_GENDER)
metrics_age       = subgroup_metrics(meta, "age_group")
metrics_race      = subgroup_metrics(meta, COL_RACE)

all_metrics = pd.concat([
    metrics_gender,
    metrics_age,
    metrics_race,
])

# Also add overall row
overall = compute_metrics(meta[COL_LABEL].values, meta["prob"].values)
overall["group_col"] = "overall"
overall["group"]     = "Overall"
all_metrics = pd.concat([pd.DataFrame([overall]).set_index("group"), all_metrics])

display_cols = ["n", "n_pos", "prevalence", "auroc", "auprc",
                "accuracy", "precision", "recall", "specificity", "f1", "fpr", "fnr"]
print(all_metrics[display_cols].round(4).to_string())

all_metrics.to_csv("results/subgroup_metrics.csv")
print("\nSaved → results/subgroup_metrics.csv")